<a href="https://colab.research.google.com/github/JGabriel-007/Projeto-UDF/blob/main/C%C3%93DIGO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bibliotecas**

In [2]:
import sqlite3 as sql
import pandas as pd
import tkinter as tk

In [3]:
import sqlite3
import os

def criar_banco():
    nome_banco = "estoque.db"

    # Verifica se o arquivo já existe
    if not os.path.exists(nome_banco):
        print("Banco não encontrado. Criando banco...")
    else:
        print("Banco já existe. Apenas abrindo conexão...")

    # Cria ou abre o banco
    conexao = sqlite3.connect(nome_banco)
    cursor = conexao.cursor()

    # Cria a tabela
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS estoque (
        codigo TEXT PRIMARY KEY,
        nome_item TEXT NOT NULL,
        quantidade REAL NOT NULL,
        unidade_medida TEXT NOT NULL,
        preco_unitario REAL NOT NULL,
        preco_total REAL NOT NULL
    )
    """)

    conexao.commit()
    conexao.close()

    print("Banco pronto para uso!")

criar_banco()

Banco não encontrado. Criando banco...
Banco pronto para uso!


In [4]:
%%writefile cadastro_produtos.py

import sqlite3

def conectar_banco():
    return sqlite3.connect("estoque.db")


def criar_tabela():
    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS estoque (
        codigo TEXT PRIMARY KEY,
        nome_item TEXT NOT NULL,
        quantidade REAL NOT NULL,
        unidade_medida TEXT NOT NULL,
        preco_unitario REAL NOT NULL,
        preco_total REAL NOT NULL
    )
    """)

    conexao.commit()
    conexao.close()


def cadastrar_produto():
    print("\n--- CADASTRO DE PRODUTO ---")

    codigo = input("Digite o código do produto (001 a 999): ")

    if not codigo.isdigit() or len(codigo) != 3:
        print("Erro: o código deve conter 3 números. Exemplo: 001.")
        return

    nome_item = input("Digite o nome do item: ")
    quantidade = float(input("Digite a quantidade em estoque: "))
    unidade_medida = input("Digite a unidade de medida (un, kg, L, m): ")
    preco_unitario = float(input("Digite o preço unitário: "))

    preco_total = quantidade * preco_unitario

    produto = {
        "codigo": codigo,
        "nome_item": nome_item,
        "quantidade": quantidade,
        "unidade_medida": unidade_medida,
        "preco_unitario": preco_unitario,
        "preco_total": preco_total
    }

    conexao = conectar_banco()
    cursor = conexao.cursor()

    try:
        cursor.execute("""
        INSERT INTO estoque (
            codigo,
            nome_item,
            quantidade,
            unidade_medida,
            preco_unitario,
            preco_total
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """, (
            produto["codigo"],
            produto["nome_item"],
            produto["quantidade"],
            produto["unidade_medida"],
            produto["preco_unitario"],
            produto["preco_total"]
        ))

        conexao.commit()
        print("\nProduto cadastrado com sucesso!")
        print(f"Preço total em estoque: R$ {preco_total:.2f}")

    except sqlite3.IntegrityError:
        print("\nErro: já existe um produto com esse código.")

    conexao.close()


def listar_produtos():
    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("SELECT * FROM estoque")
    produtos = cursor.fetchall()

    print("\n--- PRODUTOS CADASTRADOS ---")

    if len(produtos) == 0:
        print("Nenhum produto cadastrado.")
    else:
        for produto in produtos:
            print(f"""
Código: {produto[0]}
Nome: {produto[1]}
Quantidade: {produto[2]}
Unidade: {produto[3]}
Preço unitário: R$ {produto[4]:.2f}
Preço total: R$ {produto[5]:.2f}
------------------------------
""")

    conexao.close()

def remover_produto():
    print("\n--- REMOVER PRODUTO ---")
    codigo = input("Digite o código do produto a ser removido: ")

    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("DELETE FROM estoque WHERE codigo = ?", (codigo,))
    conexao.commit()

    if cursor.rowcount > 0:
        print(f"Produto com código {codigo} removido com sucesso!")
    else:
        print(f"Nenhum produto encontrado com o código {codigo}.")

    conexao.close()

def atualizar_produto():
    print("\n--- ATUALIZAR PRODUTO ---")
    codigo = input("Digite o código do produto a ser atualizado: ")

    conexao = conectar_banco()
    cursor = conexao.cursor()

    cursor.execute("SELECT * FROM estoque WHERE codigo = ?", (codigo,))
    produto = cursor.fetchone()

    if produto:
        print("\nProduto encontrado:")
        print(f"Código: {produto[0]}")
        print(f"Nome atual: {produto[1]}")
        print(f"Quantidade atual: {produto[2]}")
        print(f"Unidade de medida atual: {produto[3]}")
        print(f"Preço unitário atual: R$ {produto[4]:.2f}")

        novo_nome_item = input(f"Digite o novo nome do item (pressione Enter para manter '{produto[1]}'): ")
        nova_quantidade = input(f"Digite a nova quantidade (pressione Enter para manter '{produto[2]}'): ")
        nova_unidade_medida = input(f"Digite a nova unidade de medida (pressione Enter para manter '{produto[3]}'): ")
        novo_preco_unitario = input(f"Digite o novo preço unitário (pressione Enter para manter 'R$ {produto[4]:.2f}'): ")

        nome_item = novo_nome_item if novo_nome_item else produto[1]
        quantidade = float(nova_quantidade) if nova_quantidade else produto[2]
        unidade_medida = nova_unidade_medida if nova_unidade_medida else produto[3]
        preco_unitario = float(novo_preco_unitario) if novo_preco_unitario else produto[4]

        novo_preco_total = quantidade * preco_unitario

        cursor.execute("""
        UPDATE estoque
        SET nome_item = ?,
            quantidade = ?,
            unidade_medida = ?,
            preco_unitario = ?,
            preco_total = ?
        WHERE codigo = ?
        """, (
            nome_item,
            quantidade,
            unidade_medida,
            preco_unitario,
            novo_preco_total,
            codigo
        ))

        conexao.commit()
        print(f"\nProduto com código {codigo} atualizado com sucesso!")
        print(f"Novo preço total em estoque: R$ {novo_preco_total:.2f}")
    else:
        print(f"Nenhum produto encontrado com o código {codigo}.")

    conexao.close()

def menu():
    criar_tabela()

    while True:
        print("""
===== SISTEMA DE ESTOQUE =====
1 - Cadastrar produto
2 - Listar produtos
3 - Atualizar produto
4 - Remover produto
5 - Sair
""")

        opcao = input("Escolha uma opção: ")

        if opcao == "1":
            cadastrar_produto()
        elif opcao == "2":
            listar_produtos()
        elif opcao == "3":
            atualizar_produto()
        elif opcao == "4":
            remover_produto()
        elif opcao == "5":
            print("Sistema encerrado.")
            break
        else:
            print("Opção inválida. Tente novamente.")


menu()

Writing cadastro_produtos.py


In [5]:
!python cadastro_produtos.py


===== SISTEMA DE ESTOQUE =====
1 - Cadastrar produto
2 - Listar produtos
3 - Atualizar produto
4 - Remover produto
5 - Sair

Escolha uma opção: 2

--- PRODUTOS CADASTRADOS ---
Nenhum produto cadastrado.

===== SISTEMA DE ESTOQUE =====
1 - Cadastrar produto
2 - Listar produtos
3 - Atualizar produto
4 - Remover produto
5 - Sair

Escolha uma opção: 5
Sistema encerrado.
